# Steps 4-13 — Split, train, blend, calibrate, evaluate, export

| | |
|---|---|
| **Purpose** | Train the two models, blend and calibrate them, score test and holdout once, and export everything the site needs. |
| **Input** | `data/model_df.parquet` (from `features.ipynb`), helpers in `pl_model.py`. |
| **Output** | `models/xgb_clf.json`, `models/xgb_pois_{home,away}.json`, `models/config.json`, `data/ui_predictions.json`, `data/ui_meta.json`, `data/team_index.json`. |

Loads `data/model_df.parquet` (from `features.ipynb`) and builds the 3-class (H / D / A)
match-outcome model. Three predictors, blended and calibrated:

| # | model | what it is |
|---|---|---|
| **A** | **XGBoost classifier** | `multi:softprob`, **no class weights**, `base_margin` = the vig-free market log-odds — so the trees only learn where public stats beat the price |
| **B** | **Dixon–Coles goal model** | two `count:poisson` regressors → E[home goals], E[away goals] → bivariate-Poisson scoreline matrix with the low-score τ correction → P(H/D/A). A draw falls out of the maths instead of being a class to separate |
| **A⊕B** | **blend + temperature calibration** | weight tuned on validation, then one temperature parameter fit on validation so stated probabilities match observed frequencies (a per-class isotonic map is fitted alongside for comparison only) |

**Benchmarks, hardest first:** the **market** log loss (de-vigged Bet365 / consensus) — then
always-home (~0.46 acc). Beating the market is the real bar.

Steps: **4** chronological split · **5** baselines · **6** classifier · **7** Dixon–Coles ·
**8** blend + calibrate + decision thresholds · **9** evaluate on test · **10** feature
importance & ablation · **11** retrain + save · **12** holdout (once) · **13** export for the
interface.

> Runs on the `condaenv` kernel (Python 3.12). Metrics, isotonic calibration and the
> Dixon–Coles matrix are hand-rolled in `pl_model.py` — numpy/pandas/xgboost only, because
> Smart App Control blocks scikit-learn's native DLLs here.


In [ ]:
import json

import numpy as np
import pandas as pd
import xgboost as xgb

from pl_model import (LABELS, Y_MAP, INV, log_loss, rps, accuracy, confusion,
                      class_report, to_pred, TemperatureScale, MultiIsotonic,
                      dc_outcome_probs, fit_rho, feature_groups)
# inference helpers shared with predict_upcoming.py: the market prior as base margin, DMatrix
# construction, early-stopping-aware prediction, per-row contributions, the scoreline grid
from pl_infer import (market_base_margin, goals_base_margin, clf_dmatrix, goals_dmatrix,
                      clf_proba, poisson_pred, contribs_for, score_grid, predict_bundle, market_proba)

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
np.set_printoptions(suppress=True, linewidth=140)
print("xgboost", xgb.__version__)


## 4 — Chronological split

A random split leaks: form features carry the future backwards. We cut by season.

| slice | seasons | purpose |
|---|---|---|
| **train** | 2000-01 … 2020-21 | fit the trees |
| **valid** | 2021-22 + 2022-23 | early stopping, blend weight, Dixon–Coles ρ, calibration |
| **test**  | 2023-24 + 2024-25 | honest iteration |
| **holdout** | 2025-26 + 2026-27 | run once, at the very end |

Keep matches where both teams have ≥ 3 games of history this season (cold-start rows have
no meaningful form).


In [ ]:
df = pd.read_parquet("data/model_df.parquet")
df = df[(df.home_games_played >= 3) & (df.away_games_played >= 3)].copy()
df = df.sort_values("Date").reset_index(drop=True)
df["y"] = df.target.map(Y_MAP)

VALID = ["2021-22", "2022-23"]
TEST = ["2023-24", "2024-25"]
HOLDOUT = ["2025-26", "2026-27"]

train = df[~df.Season.isin(VALID + TEST + HOLDOUT)].copy()
valid = df[df.Season.isin(VALID)].copy()
test = df[df.Season.isin(TEST)].copy()
holdout = df[df.Season.isin(HOLDOUT)].copy()
trainval = pd.concat([train, valid])                     # for final refits

G = feature_groups(df)
for k, v in G.items():
    print(f"{k:7s} {len(v):3d} features")

for name, part in [("train", train), ("valid", valid), ("test", test), ("holdout", holdout)]:
    cov = part[["mkt_p_H", "mkt_p_D", "mkt_p_A"]].notna().all(axis=1).mean()
    print(f"{name:8s} {len(part):5d}   {part.Season.min()} .. {part.Season.max()}   market cov {cov:.0%}")
print("\ntrain target mix %:", (train.target.value_counts(normalize=True) * 100).round(1).to_dict())

# market_base_margin / goals_base_margin come from pl_infer: the trees start from the market's price.


## 5 — Baselines

- **always-home** — predicts H every time (~0.46 accuracy on the PL).
- **market (Shin)** — the de-vigged consensus price. This is the number to beat: a fair
  bookmaker's closing line has a log loss around **0.95–0.97** on recent seasons.
- **market (proportional)** — the cruder de-vig, for comparison.

Reported on every slice: accuracy, log loss, RPS (ranked probability score — the proper
rule for ordered outcomes).


In [ ]:
def scores(name, y_true, proba):
    proba = np.asarray(proba, float)
    return {"model": name, "n": len(y_true),
            "acc": accuracy(y_true, to_pred(proba)),
            "logloss": log_loss(y_true, proba),
            "rps": rps(y_true, proba)}

# market_proba(frame, "p" | "pow") comes from pl_infer: proportional vig-free, or fav-longshot corrected


rows = []
for nm, part in [("valid", valid), ("test", test)]:
    yt_ = part.target.to_numpy()
    always_home = np.tile([0.999, 0.0005, 0.0005], (len(part), 1))
    rows.append({**scores("always-home", yt_, always_home), "slice": nm})
    rows.append({**scores("market (vig-free)", yt_, market_proba(part, "p")), "slice": nm})
    rows.append({**scores("market (power)", yt_, market_proba(part, "pow")), "slice": nm})

BASELINES = pd.DataFrame(rows).set_index(["slice", "model"]).round(4)
print(BASELINES.to_string())
MARKET_TEST_LL = BASELINES.loc[("test", "market (vig-free)"), "logloss"]
print("\n>>> target to beat on test: log loss < %.4f" % MARKET_TEST_LL)


## 6 — Model A: the XGBoost classifier

Changes from the first version:

- **No `sample_weight`.** Inverse-frequency weighting was pushing draw probabilities away
  from the true base rate — on the old holdout it predicted 90 draws at 28% precision and
  the log loss collapsed to 1.07. Boosting already minimises multiclass log loss directly;
  we let the probabilities be honest and handle any draw-recall need at the decision step.
- **`base_margin` = market log-odds.** The trees start from the vig-free price and only
  need to model the *residual* — where rolling form, xG, rest and squad value disagree with
  the market. Where no odds exist (2000–02) the prior is uniform.
- **All feature groups** — form + EWMA + xG proxy + margin-aware Elo + league position +
  market-derived + FPL squad value (XGBoost splits on NaN natively, so the 8%-covered
  squad columns cost nothing on older rows).
- Shallow trees (`max_depth=3`), slow `eta=0.03`, up to 4000 rounds with **early stopping**
  on the 2-season validation block.


In [ ]:
CLF_PARAMS = dict(
    objective="multi:softprob", num_class=3, eval_metric="mlogloss",
    max_depth=3, eta=0.03, subsample=0.85, colsample_bytree=0.7,
    min_child_weight=8, reg_lambda=2.0, reg_alpha=0.5, seed=0, nthread=-1,
)
FEATS = G["all"]


def train_classifier(tr, feats, va=None, num_round=4000, params=CLF_PARAMS):
    dtr = clf_dmatrix(tr, feats, label=tr.y.to_numpy())
    evals = [(dtr, "train")]
    kw = {}
    if va is not None:
        evals.append((clf_dmatrix(va, feats, label=va.y.to_numpy()), "valid"))
        kw["early_stopping_rounds"] = 150
    return xgb.train(params, dtr, num_round, evals=evals, verbose_eval=False, **kw)


clf = train_classifier(train, FEATS, va=valid)
print("classifier: %d features, %d trees on top of the market prior"
      % (len(FEATS), (getattr(clf, "best_iteration", 0) or 0) + 1))

p_clf_valid = clf_proba(clf, valid, FEATS)
p_clf_test = clf_proba(clf, test, FEATS)
print("classifier      ", {k: round(v, 4) for k, v in scores("c", test.target.to_numpy(), p_clf_test).items() if k in ("acc", "logloss", "rps")})
print("market (vig-free)", {k: round(v, 4) for k, v in scores("m", test.target.to_numpy(), market_proba(test, "p")).items() if k in ("acc", "logloss", "rps")})


## 7 — Model B: Dixon–Coles goal model

A draw isn't a tactic — it's what happens when two low Poisson means collide (0-0, 1-1).
So instead of asking a flat classifier to carve the draw region out of overlapping
distributions, we **model the goals** and let the outcome probabilities fall out.

1. Two `count:poisson` XGBoost regressors predict `E[home goals] = λ` and
   `E[away goals] = μ`, each starting from `base_margin = log(market implied xG)`.
2. Build the scoreline matrix `P(i,j) = Poisson(i;λ)·Poisson(j;μ)·τ(i,j)`, where the
   **Dixon–Coles τ** nudges the four lowest scores to fix independent-Poisson's known
   under-prediction of 0-0 and 1-1. ρ is fit on validation.
3. Sum the matrix: `P(H)=Σ_{i>j}`, `P(D)=Σ_{i=j}`, `P(A)=Σ_{i<j}`.

The same matrix gives us an exact-score distribution for the interface's scoreline grid.


In [ ]:
POIS_PARAMS = dict(
    objective="count:poisson", eval_metric="poisson-nloglik",
    max_depth=4, eta=0.03, subsample=0.85, colsample_bytree=0.7,
    min_child_weight=8, reg_lambda=2.0, max_delta_step=0.7, seed=0, nthread=-1,
)


def train_poisson(tr, feats, side, va=None, num_round=3000):
    label = "home_goals" if side == "home" else "away_goals"
    dtr = goals_dmatrix(tr, feats, side, label=tr[label].to_numpy(float))
    evals = [(dtr, "train")]
    kw = {}
    if va is not None:
        evals.append((goals_dmatrix(va, feats, side, label=va[label].to_numpy(float)), "valid"))
        kw["early_stopping_rounds"] = 150
    return xgb.train(POIS_PARAMS, dtr, num_round, evals=evals, verbose_eval=False, **kw)


pois_home = train_poisson(train, FEATS, "home", va=valid)
pois_away = train_poisson(train, FEATS, "away", va=valid)
print("poisson best_iter home/away:",
      getattr(pois_home, "best_iteration", None), getattr(pois_away, "best_iteration", None))

lam_valid = poisson_pred(pois_home, valid, FEATS, "home")
mu_valid = poisson_pred(pois_away, valid, FEATS, "away")
lam_test = poisson_pred(pois_home, test, FEATS, "home")
mu_test = poisson_pred(pois_away, test, FEATS, "away")

RHO, rho_ll = fit_rho(lam_valid, mu_valid, valid.target.to_numpy())
print(f"\nDixon-Coles rho = {RHO:+.3f}  (valid outcome log loss {rho_ll:.4f})")
print(f"mean predicted goals  home {lam_test.mean():.2f}  away {mu_test.mean():.2f}   "
      f"actual  home {test.home_goals.mean():.2f}  away {test.away_goals.mean():.2f}")

p_dc_valid = dc_outcome_probs(lam_valid, mu_valid, RHO)
p_dc_test = dc_outcome_probs(lam_test, mu_test, RHO)
print("dixon-coles ",
      {k: round(v, 4) for k, v in scores("dc", test.target.to_numpy(), p_dc_test).items()
       if k not in ("model", "n")})


## 8 — Blend + temperature calibration + decision threshold

- **Blend.** `p = w · A + (1−w) · B`. The two models make different errors — the
  classifier is sharper on strong favourites, Dixon–Coles on tight low-scoring games — so
  a weighted average beats both. `w` is the value that minimises validation log loss.
- **Calibration.** Even a good log loss can be mis-calibrated (says 60%, happens 52%). One
  **temperature** parameter `T`, fit on validation, rescales the blend's log-probabilities so
  the stated chance matches the observed rate. A per-class isotonic map is fitted alongside for
  comparison; on a 700-match validation block it overfits, so temperature is what ships.
- **Decision threshold.** `argmax` is the log-loss-optimal pick. If a use case wants more
  draws called (betting, tactical sims), we search a small utility matrix on validation —
  reported alongside, never fed back into training.


In [ ]:
yv = valid.target.to_numpy()

ws = np.linspace(0, 1, 41)
ll_by_w = [log_loss(yv, w * p_clf_valid + (1 - w) * p_dc_valid) for w in ws]
W = float(ws[int(np.argmin(ll_by_w))])
print("blend weight w=%.2f  (classifier %.0f%% / dixon-coles %.0f%%)   valid ll %.4f  vs  clf %.4f  dc %.4f"
      % (W, W * 100, (1 - W) * 100, min(ll_by_w), log_loss(yv, p_clf_valid), log_loss(yv, p_dc_valid)))


def blend(pc, pd_):
    return W * pc + (1 - W) * pd_


p_blend_valid = blend(p_clf_valid, p_dc_valid)
p_blend_test = blend(p_clf_test, p_dc_test)

TEMP = TemperatureScale().fit(p_blend_valid, yv)
ISO = MultiIsotonic().fit(p_blend_valid, yv)
print("temperature T = %.2f" % TEMP.T)

CAL = TEMP
p_cal_test = CAL.transform(p_blend_test)

comp = pd.DataFrame([
    scores("A  classifier", test.target.to_numpy(), p_clf_test),
    scores("B  dixon-coles", test.target.to_numpy(), p_dc_test),
    scores("A+B  blend", test.target.to_numpy(), p_blend_test),
    scores("A+B  blend + temp", test.target.to_numpy(), TEMP.transform(p_blend_test)),
    scores("A+B  blend + isotonic", test.target.to_numpy(), ISO.transform(p_blend_test)),
    scores("market (vig-free)", test.target.to_numpy(), market_proba(test, "p")),
]).set_index("model").round(4)
print("\nTEST 2023-24 + 2024-25:")
print(comp.to_string())

# --- optional draw-seeking decision: bump D until its valid recall hits a target
def utility_decision(proba, dbonus):
    w = np.array([1.0, float(dbonus), 1.0])
    return np.array([INV[i] for i in (proba * w).argmax(1)])


DRAW_BONUS = 1.0
for b in np.linspace(1.0, 2.4, 29):
    if class_report(yv, utility_decision(p_blend_valid, b)).loc["D", "recall"] >= 0.40:
        DRAW_BONUS = round(float(b), 2)
        break
crd = class_report(test.target.to_numpy(), utility_decision(p_cal_test, DRAW_BONUS)).loc["D"]
print("\ndraw-seeking mode: multiply P(draw) by %.2f before argmax" % DRAW_BONUS)
print("   argmax  test:  draw recall %.2f  precision %.2f  (it never calls a draw)"
      % (0.0, 0.0))
print("   utility test:  draw recall %.2f  precision %.2f" % (crd["recall"], crd["precision"]))


### 8b — the pipeline as one object

`fit_bundle(fit_df, tune_df)` = classifier + two Poisson models + blend weight `W` + Dixon–Coles `ρ` + temperature `T`, each meta-parameter tuned on `tune_df`. `predict_bundle` (from `pl_infer.py`, shared with `predict_upcoming.py`) returns A, B, the blend and the calibrated blend for any slice. Used for the rolling-origin check and the two final refits.

In [ ]:
def fit_bundle(fit_df, tune_df, feats=FEATS):
    clf_ = train_classifier(fit_df, feats, va=tune_df)
    ph_ = train_poisson(fit_df, feats, "home", va=tune_df)
    pa_ = train_poisson(fit_df, feats, "away", va=tune_df)
    lam_t = poisson_pred(ph_, tune_df, feats, "home")
    mu_t = poisson_pred(pa_, tune_df, feats, "away")
    rho_, _ = fit_rho(lam_t, mu_t, tune_df.target.to_numpy())
    pc_t, pd_t = clf_proba(clf_, tune_df, feats), dc_outcome_probs(lam_t, mu_t, rho_)
    ws_ = np.linspace(0, 1, 41)
    w_ = float(ws_[int(np.argmin([log_loss(tune_df.target.to_numpy(), w * pc_t + (1 - w) * pd_t) for w in ws_]))])
    cal_ = TemperatureScale().fit(w_ * pc_t + (1 - w_) * pd_t, tune_df.target.to_numpy())
    return dict(clf=clf_, pois_home=ph_, pois_away=pa_, rho=rho_, w=w_, cal=cal_, feats=feats)


# predict_bundle(B, frame) comes from pl_infer and returns lam, mu, clf, dc, blend and cal for any slice.
print("fit_bundle ready; predict_bundle imported from pl_infer")


## 9 — Evaluate on test (full picture)

Per-class precision / recall / F1, the confusion matrix (rows = truth), and a calibration
check: bucket the predicted P(home win) into deciles and compare to the realised home-win
rate in each bucket.


In [ ]:
yt = test.target.to_numpy()
pred_cal = to_pred(p_cal_test)
base = (test.target == "H").mean()

print(f"=== TEST (n={len(test)})  —  final model = blend + calibration ===")
print(f"accuracy {accuracy(yt, pred_cal):.3f}   always-home {base:.3f}   lift {accuracy(yt, pred_cal) - base:+.3f}")
print(f"log loss {log_loss(yt, p_cal_test):.4f}   market {log_loss(yt, market_proba(test,'p')):.4f}   "
      f"delta {log_loss(yt, p_cal_test) - log_loss(yt, market_proba(test,'p')):+.4f}")
print(f"RPS      {rps(yt, p_cal_test):.4f}   market {rps(yt, market_proba(test,'p')):.4f}\n")
print(class_report(yt, pred_cal).round(3).to_string())
print()
print(confusion(yt, pred_cal).to_string())

# calibration of P(home win), deciles
ph = p_cal_test[:, 0]
bins = pd.qcut(ph, 10, duplicates="drop")
cal = pd.DataFrame({"pred_pH": ph, "is_H": (test.target.to_numpy() == "H").astype(int)})
cal_tbl = cal.groupby(bins, observed=True).agg(n=("is_H", "size"),
                                               pred=("pred_pH", "mean"),
                                               actual=("is_H", "mean")).round(3)
print("\nP(home win) calibration by decile:")
print(cal_tbl.to_string())
print(f"\nmean |pred - actual| across buckets: {(cal_tbl.pred - cal_tbl.actual).abs().mean():.3f}")


## 10 — Feature importance & ablation

- **Gain importance** of the classifier — expect `elo_diff`, market columns, season-to-date
  goal difference near the top.
- **Ablation** — does each feature group actually pull its weight? Retrain the classifier
  on `form only`, `+ market`, `+ squad`, `+ xG`, compare test log loss.
- **Rolling-origin** — retrain for each recent season on only what preceded it; look at the
  spread, not one lucky split.


In [ ]:
imp = pd.Series(clf.get_score(importance_type="gain")).sort_values(ascending=False)
print("classifier gain importance, top 20")
print(imp.head(20).round(2).to_string())

# --- ablation --------------------------------------------------------------------
xg_cols = [c for c in G["form"] if "xg" in c or "finishing" in c or "def_luck" in c]
sets = {
    "form only":        [c for c in G["form"] if c not in xg_cols] + G["elo"],
    "form + xG":         G["form"] + G["elo"],
    "+ market":          G["form"] + G["elo"] + G["market"],
    "+ squad  (=ALL)":   G["all"],
}
abl = []
for name, feats in sets.items():
    m = train_classifier(train, feats, va=valid)
    pt = clf_proba(m, test, feats)
    abl.append({"feature set": name, "n_feats": len(feats),
                "test_logloss": round(log_loss(yt, pt), 4),
                "test_acc": round(accuracy(yt, to_pred(pt)), 3)})
print("\nablation (classifier only, test):")
print(pd.DataFrame(abl).to_string(index=False))


### 10b — Rolling-origin check

One split can be lucky. For each recent season, retrain the full blend on everything
before it and score that season alone.


In [ ]:
seasons_all = sorted(df.Season.unique())
roll = []
for s in ["2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]:
    prev = [x for x in seasons_all if x < s][-2:]          # 2 seasons before the target
    tr = df[df.Season < prev[0]]                            # strictly before the tune block
    tu = df[df.Season.isin(prev)]
    te = df[df.Season == s]
    if len(tr) < 1500 or len(te) == 0:
        continue
    B = fit_bundle(tr, tu)
    pr = predict_bundle(B, te)
    yy = te.target.to_numpy()
    roll.append({"season": s, "n": len(te),
                 "acc": accuracy(yy, to_pred(pr["cal"])), "base": (te.target == "H").mean(),
                 "model_ll": log_loss(yy, pr["cal"]), "market_ll": log_loss(yy, market_proba(te, "p")),
                 "A_ll": log_loss(yy, pr["clf"])})

roll = pd.DataFrame(roll).set_index("season").round(3)
roll["edge"] = (roll.market_ll - roll.model_ll).round(3)
print(roll.to_string())
print("\nmean:", roll[["acc", "base", "model_ll", "market_ll", "A_ll"]].mean().round(3).to_dict())
print("beats market log loss in %d/%d seasons; mean edge %+.4f"
      % ((roll.model_ll < roll.market_ll).sum(), len(roll), roll.edge.mean()))


## 11 — Package the pipeline & retrain

`fit_bundle(fit_df, tune_df)` trains everything and returns one object:
classifier + two Poisson models + blend weight `W` + Dixon–Coles `ρ` + the temperature
calibrator. `predict_bundle` runs A, B, the blend, and the calibrated blend for any slice.

Two instances, each tuned on data that comes **before** what it will be scored on:

| bundle | fit on | tuned on | scores |
|---|---|---|---|
| `BT` | train (…2020-21) | valid (21-22, 22-23) | test |
| `BH` | train + valid (…2022-23) | test (23-24, 24-25) | **holdout** |


In [ ]:
import os
os.makedirs("models", exist_ok=True)

BT = fit_bundle(train, valid)          # tuned on 2021-23, scores 2023-25
BH = fit_bundle(trainval, test)        # tuned on 2023-25, scores the holdout
print("BT  w=%.2f rho=%+.3f T=%.2f    BH  w=%.2f rho=%+.3f T=%.2f"
      % (BT["w"], BT["rho"], BT["cal"].T, BH["w"], BH["rho"], BH["cal"].T))

BH["clf"].save_model("models/xgb_clf.json")
BH["pois_home"].save_model("models/xgb_pois_home.json")
BH["pois_away"].save_model("models/xgb_pois_away.json")
CONFIG = {
    "features": FEATS,
    "blend_weight_w": BH["w"], "dixon_coles_rho": BH["rho"], "temperature": BH["cal"].T,
    "clf_params": CLF_PARAMS, "pois_params": POIS_PARAMS,
    "splits": {"valid": VALID, "test": TEST, "holdout": HOLDOUT},
    "draw_utility_bonus": float(DRAW_BONUS),
}
json.dump(CONFIG, open("models/config.json", "w"), indent=1)
print("saved models/xgb_clf.json, xgb_pois_{home,away}.json, config.json")


## 12 — Final holdout — run ONCE

`BH` was fit on 2000-01…2022-23 and tuned on 2023-24…2024-25. It has never seen 2025-26 or
2026-27. This is the project's honest number — do not tweak-and-rerun.


In [ ]:
RUN_HOLDOUT = True

if RUN_HOLDOUT:
    yh = holdout.target.to_numpy()
    ph_out = predict_bundle(BH, holdout)
    mkt_h = market_proba(holdout, "p")
    tbl = pd.DataFrame([
        scores("always-home", yh, np.tile([0.999, 5e-4, 5e-4], (len(yh), 1))),
        scores("market (vig-free)", yh, mkt_h),
        scores("A  classifier", yh, ph_out["clf"]),
        scores("B  dixon-coles", yh, ph_out["dc"]),
        scores("A+B blend + calib", yh, ph_out["cal"]),
    ]).set_index("model").round(4)
    print("=== HOLDOUT 2025-26 + 2026-27  (n=%d) ===" % len(holdout))
    print(tbl.to_string())
    print()
    print(class_report(yh, to_pred(ph_out["cal"])).round(3).to_string())
    print()
    print(confusion(yh, to_pred(ph_out["cal"])).to_string())
    edge = tbl.loc["market (vig-free)", "logloss"] - tbl.loc["A+B blend + calib", "logloss"]
    print("\nlog-loss vs the market: %+.4f   (%s)"
          % (edge, "ahead" if edge > 0 else "behind — the closing line is hard to beat"))
else:
    print("holdout skipped")


## 13 — Export for the interface

Writes `data/ui_predictions.json` — one record per match from 2023-24 onward:
blind prediction (probabilities, most likely score from the Dixon–Coles grid, top feature
signals), the actual result, and whether the pick was right. Test-season matches are
predicted by `BT`, holdout matches by `BH` — each model blind to the season it is scoring.
Plus `data/ui_meta.json` (headline metrics, feature importance, calibration curve) and
`data/team_index.json` (each club's latest Elo & form for the "custom matchup" mode).


In [ ]:
# contribs_for() and score_grid() come from pl_infer, so the site's season page is built the same way.
records = []
for seasons, B, tag in [(TEST, BT, "test"), (HOLDOUT, BH, "holdout")]:
    part = df[df.Season.isin(seasons)].reset_index(drop=True)
    pr = predict_bundle(B, part)
    ctr = contribs_for(B["clf"], part, B["feats"])
    mkt = market_proba(part, "p")
    for i, r in part.iterrows():
        cal = pr["cal"][i]
        pick = LABELS[int(cal.argmax())]
        played = pd.notna(r.FTHG) and pd.notna(r.FTAG)
        records.append({
            "id": int(r.match_id), "season": r.Season, "date": r.Date.strftime("%Y-%m-%d"),
            "round": int(r.home_match_round) if pd.notna(r.home_match_round) else None,
            "home": r.HomeTeam, "away": r.AwayTeam, "eval_group": tag,
            "played": bool(played),
            "actual": (r.FTR if played else None),
            "score": ([int(r.FTHG), int(r.FTAG)] if played else None),
            "pred": pick,
            "proba": {"H": round(float(cal[0]), 4), "D": round(float(cal[1]), 4), "A": round(float(cal[2]), 4)},
            "proba_clf": {k: round(float(v), 4) for k, v in zip(LABELS, pr["clf"][i])},
            "proba_dc": {k: round(float(v), 4) for k, v in zip(LABELS, pr["dc"][i])},
            "market": ({"H": round(float(mkt[i][0]), 4), "D": round(float(mkt[i][1]), 4),
                        "A": round(float(mkt[i][2]), 4)} if np.isfinite(mkt[i]).all() else None),
            "xg": {"home": round(float(pr["lam"][i]), 2), "away": round(float(pr["mu"][i]), 2)},
            "elo": {"home": round(float(r.home_elo_pre)), "away": round(float(r.away_elo_pre))},
            "grid": score_grid(float(pr["lam"][i]), float(pr["mu"][i]), B["rho"]),
            "signals": ctr[i],
            "correct": (bool(pick == r.FTR) if played else None),
        })

with open("data/ui_predictions.json", "w") as fh:
    json.dump(records, fh)
print(f"wrote data/ui_predictions.json  ({len(records)} matches, "
      f"{sum(x['played'] for x in records)} played)")

# --- team index for the custom-matchup mode ----------------------------------
ti = {}
for t in sorted(set(df.HomeTeam) | set(df.AwayTeam)):
    sub = df[(df.HomeTeam == t) | (df.AwayTeam == t)].sort_values("Date")
    if not len(sub):
        continue
    last = sub.iloc[-1]
    home = last.HomeTeam == t
    ti[t] = {
        "season": last.Season,
        "elo": round(float(last.home_elo_pre if home else last.away_elo_pre)),
        "pts_ewm": round(float(last.home_pts_ewm if home else last.away_pts_ewm), 2),
        "xgd_ewm": round(float(last.home_xgd_ewm if home else last.away_xgd_ewm), 2),
        "rank": int(last.home_rank if home else last.away_rank) if pd.notna(last.home_rank) else None,
    }
with open("data/team_index.json", "w") as fh:
    json.dump(ti, fh, indent=1)
print(f"wrote data/team_index.json  ({len(ti)} clubs)")

# --- headline metrics + importance + calibration ----------------------------
def metric_block(part, B):
    y = part.target.to_numpy(); pr = predict_bundle(B, part); m = market_proba(part, "p")
    return {
        "n": len(part),
        "model": {k: round(v, 4) for k, v in scores("m", y, pr["cal"]).items() if k in ("acc", "logloss", "rps")},
        "market": {k: round(v, 4) for k, v in scores("m", y, m).items() if k in ("acc", "logloss", "rps")},
        "baseline_acc": round(float((part.target == "H").mean()), 4),
    }

meta = {
    "trained_matches": int(len(trainval)), "total_matches": int(len(df)),
    "n_features": len(FEATS), "blend_w": BH["w"], "dixon_coles_rho": BH["rho"],
    "test": metric_block(test, BT), "holdout": metric_block(holdout, BH),
    "importance": [{"feat": k, "label": label_of(k), "gain": round(float(v), 1)}
                   for k, v in imp.head(22).items()],
    "calibration": [{"pred": float(p), "actual": float(a), "n": int(n)}
                    for p, a, n in zip(cal_tbl.pred, cal_tbl.actual, cal_tbl.n)],
    "rolling_origin": roll.reset_index().to_dict("records"),
    "outcome_dist": {k: round(float(v), 3) for k, v in df.target.value_counts(normalize=True).items()},
}
with open("data/ui_meta.json", "w") as fh:
    json.dump(meta, fh, indent=1)
print("wrote data/ui_meta.json")
print(json.dumps(meta["test"], indent=1))
